# Day 6: TF-CoVR (AIM) features for FineGym vault clips

**Runtime → Change runtime type → T4 GPU**, upload `aim_model.py` to the Colab file panel (left), then *Runtime → Run all*.

Same clips, same windows (whole / approach / flight / landing) and same output format as the Day 3 VideoMAE notebook. Only the backbone changes: the AIM encoder from **TF-CoVR** (CLIP ViT-B/16 with temporal adapters, fine-tuned by the TF-CoVR authors to classify 306 FineGym + FineDiving sub-actions, 12 frames at 224×224).

Outputs on Drive, in `capstone/features/`:

| file | contents |
|---|---|
| `aim_tfcovr_all.npz` | `whole` 768, `whole_seq` 12×768, `stages` 3×768, `has_stages`, `pred_class` |
| `aim_tfcovr_index.csv` | clip metadata, same row order |
| `day6_summary.txt` | sanity checks, compared with VideoMAE |

and `capstone/tfcovr_annotations/` (TF-CoVR's train/val/test CSVs, for the overlap check).

Resumable: if Colab disconnects, *Run all* again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install gdown einops huggingface_hub

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
ROOT      = Path('/content/drive/MyDrive/capstone')
CLIPS_DRV = ROOT / 'clips'
CSV       = ROOT / 'out'/'clips_available.csv'
FEAT_DIR  = ROOT / 'features' / 'aim_tfcovr'
CKPT_DIR  = ROOT / 'checkpoints'
ANN_DIR   = ROOT / 'tfcovr_annotations'
for d in (FEAT_DIR, CKPT_DIR, ANN_DIR):
    d.mkdir(parents=True, exist_ok=True)
LOCAL     = Path('/content/clips')

assert CSV.exists(), f'missing {CSV}'
assert Path('/content/aim_model.py').exists(), 'upload aim_model.py to the Colab file panel first'
sys.path.insert(0, '/content')
print('ok')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ok


## 2. Download the TF-CoVR stage-1 weights and annotation CSVs
Weights: the "stage 1 pretrained weights" Google Drive link from the TF-CoVR README, saved once to your Drive.
Annotations: only the CSV files from the TF-CoVR Hugging Face dataset (not the embeddings).

In [ ]:
import gdown
CKPT = CKPT_DIR / 'tfcovr_aim_stage1.ckpt'
if not CKPT.exists():
    gdown.download(id='14MSr6PxPhKgriF3PzA5DKp8X9sjUu_WK', output=str(CKPT), quiet=False)
print(CKPT, f'{CKPT.stat().st_size / 1e6:.0f} MB')

from huggingface_hub import snapshot_download
import shutil
p = snapshot_download('ucf-crcv/TF-CoVR', repo_type='dataset', allow_patterns=['*.csv'])
csvs = list(Path(p).rglob('*.csv'))
for f in csvs:
    shutil.copy(f, ANN_DIR / f.name)
print('TF-CoVR CSVs:', [f.name for f in csvs])

Downloading...
From (original): https://drive.google.com/uc?id=14MSr6PxPhKgriF3PzA5DKp8X9sjUu_WK
From (redirected): https://drive.google.com/uc?id=14MSr6PxPhKgriF3PzA5DKp8X9sjUu_WK&confirm=t&uuid=353b2918-fa99-44d8-92a5-0bb43e05dd32
To: /content/drive/MyDrive/capstone/checkpoints/tfcovr_aim_stage1.ckpt
100%|██████████| 1.16G/1.16G [00:18<00:00, 62.0MB/s]


/content/drive/MyDrive/capstone/checkpoints/tfcovr_aim_stage1.ckpt 1161 MB


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

TF-CoVR CSVs: ['test_finegd.csv', 'train_finegd.csv', 'val_finegd.csv']


## 3. Clip list (train + test) and local copy

In [ ]:
import pandas as pd
df = pd.read_csv(CSV, dtype={'vault_code': str, 'bib': str})
df = df[df.split.isin(['train', 'test'])].reset_index(drop=True)
df['clip_id'] = df.video_id + '__' + df.event_key
print(df.split.value_counts().to_dict(), '| clips without stages:', df.segment_key.isna().sum())

LOCAL.mkdir(exist_ok=True)
copied = 0
for f in df.clip_file:
    if not (LOCAL / f).exists():
        shutil.copy(CLIPS_DRV / f, LOCAL / f); copied += 1
print('copied', copied, '| local clips:', len(list(LOCAL.glob('*.mp4'))))

{'train': 784, 'test': 177} | clips without stages: 75
copied 961 | local clips: 961


## 4. Video reading and windows (identical to Day 3, except 12 frames per window)
TF-CoVR samples 12 frames at the centres of 12 equal bins and squashes each frame to 224×224 with CLIP normalisation; this does the same.

In [ ]:
import json, math
import cv2
import numpy as np

N_FRAMES = 12  # AIM input length


def read_clip(path, short_side=256):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        h, w = f.shape[:2]
        s = short_side / min(h, w)
        f = cv2.resize(f, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, fps


def sample_window(frames, fps, t0, t1, n=N_FRAMES):
    last = len(frames) - 1
    t1 = max(t1, t0 + 1e-3)
    times = [t0 + (i + 0.5) * (t1 - t0) / n for i in range(n)]
    idx = [min(last, max(0, int(math.floor(t * fps)))) for t in times]
    return [frames[i] for i in idx], idx


def windows_for_row(row):
    off = float(row['stage_offset'])
    whole = (off, off + float(row['duration']))
    stages = None
    raw = row.get('stages')
    if isinstance(raw, str) and raw.strip().startswith('['):
        st = json.loads(raw)
        if len(st) == 3:
            stages = [(off + a, off + b) for a, b in st]
    return whole, stages

## 5. Model
Loads the weights and **stops with an error if they do not match the model** (the original TF-CoVR script would silently continue with random weights). Expect "matched 100%" or close.

In [ ]:
!git clone https://github.com/ahmed-eldawy/TF-CoVR.git /content/TF-CoVR

Cloning into '/content/TF-CoVR'...
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!find /content -maxdepth 3 -type d -name "src" -o -type f -name "*.py" | head -100

/content/TF-CoVR/AIM_Embeddings/src
/content/TF-CoVR/AIM_Embeddings/aim_emb.py
/content/TF-CoVR/train.py
/content/TF-CoVR/data/frames_videos.py
/content/TF-CoVR/data/trim_subaction.py
/content/TF-CoVR/data/trim_events.py
/content/TF-CoVR/src
/content/TF-CoVR/test.py
/content/drive/MyDrive/Banana_Problem_AakashDesai.py
/content/drive/MyDrive/simple_decorators.py
/content/drive/MyDrive/calci (1).py
/content/drive/MyDrive/calci.py
/content/aim_model.py


In [ ]:
with open("/content/aim_model.py", "r") as f:
    text = f.read()

print(text[:20000])

"""AIM video encoder used by TF-CoVR (NeurIPS 2025), copied for feature extraction only.

Source: github.com/UCF-CRCV/TF-CoVR, AIM_Embeddings/src/model/aim/aim_model.py (MIT licence).
It is a CLIP ViT-B/16 image encoder with temporal, spatial and MLP adapters, fine-tuned by the
TF-CoVR authors to classify 306 FineGym + FineDiving sub-actions from 12-frame clips.

Changes from the original, none of which affect the weights:
  * DropPath replaced by Identity (it does nothing at inference) and no `clip`/`timm` imports
  * `features()` returns the pooled 768-d vector (the input of the classifier, i.e. what the
    TF-CoVR `aim_emb.py` script saves), the per-frame CLS tokens, and the 306 class logits
"""
from collections import OrderedDict

import torch
from einops import rearrange
from torch import nn


class Adapter(nn.Module):
    def __init__(self, D_features, mlp_ratio=0.25, act_layer=nn.GELU, skip_connect=True):
        super().__init__()
        self.skip_connect = skip_connect
     

In [ ]:
!wget -q https://github.com/UCF-CRCV/TF-CoVR/archive/refs/heads/main.zip -O /content/tfcovr.zip
!unzip -q -o /content/tfcovr.zip -d /content/
!mv /content/TF-CoVR-main /content/TF-CoVR

In [ ]:
# Make sure the TF-CoVR repository is actually on Python's path
import os
import sys

print(os.listdir("/content/TF-CoVR")[:20])

# Find where the repository's src directory actually is
!find /content/TF-CoVR -maxdepth 3 -type d -name "src"

['AIM_Embeddings', 'train.py', 'tools', 'requirements.txt', 'data', 'configs', 'LICENSE', 'README.md', 'TF-CoVR-main', 'src', 'test.py', 'assets', '.gitignore']
/content/TF-CoVR/AIM_Embeddings/src
/content/TF-CoVR/TF-CoVR-main/AIM_Embeddings/src
/content/TF-CoVR/TF-CoVR-main/src
/content/TF-CoVR/src


In [ ]:
import sys
sys.path.insert(0, "/content/TF-CoVR/AIM_Embeddings")

import src
print("src OK:", src.__file__)

src OK: None


In [ ]:
import torch
from aim_model import load_aim

DEV = 'cuda'
model = load_aim(CKPT, DEV)
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=DEV).view(1, 1, 3, 1, 1)
STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=DEV).view(1, 1, 3, 1, 1)


@torch.no_grad()
def embed(videos):
    """videos: list of clips, each a list of 12 RGB frames -> pooled [B,768], seq [B,12,768], class [B]"""
    x = np.stack([[cv2.resize(f, (224, 224), interpolation=cv2.INTER_CUBIC) for f in v] for v in videos])
    x = torch.from_numpy(x).to(DEV).permute(0, 1, 4, 2, 3).float() / 255.0   # [B, T, 3, 224, 224]
    pooled, seq, logits = model.features((x - MEAN) / STD)
    return pooled.cpu().numpy(), seq.cpu().numpy(), logits.argmax(1).cpu().numpy()


r = df.iloc[0]
fr, fps = read_clip(LOCAL / r.clip_file)
p, s, c = embed([sample_window(fr, fps, *windows_for_row(r)[0])[0]])
print(r.clip_id, len(fr), 'frames @', fps, 'fps ->', p.shape, s.shape, '| predicted class', c[0])

checkpoint: 298 tensors | prefix 'model.' | matched 298/298 model tensors (100.0%)
1JsRXIoR3C0__E_000598_000605 202 frames @ 25.0 fps -> (1, 768) (1, 12, 768) | predicted class 11


## 6. Extract (resumable, saves every 25 clips)

In [ ]:
import time
from tqdm.auto import tqdm

done = set()
for f in sorted(FEAT_DIR.glob('part_*.npz')):
    done.update(np.load(f, allow_pickle=True)['ids'].tolist())
todo = df[~df.clip_id.isin(done)]
print(f'{len(done)} already done, {len(todo)} to go')

KEYS = ['ids', 'whole', 'whole_seq', 'stages', 'has_stages', 'pred_class', 'n_frames', 'fps']
buf = {k: [] for k in KEYS}
part = len(list(FEAT_DIR.glob('part_*.npz')))
errors = []


def flush():
    global part, buf
    if not buf['ids']:
        return
    np.savez_compressed(FEAT_DIR / f'part_{part:03d}.npz',
        ids=np.array(buf['ids']), whole=np.stack(buf['whole']).astype(np.float16),
        whole_seq=np.stack(buf['whole_seq']).astype(np.float16),
        stages=np.stack(buf['stages']).astype(np.float16), has_stages=np.array(buf['has_stages']),
        pred_class=np.array(buf['pred_class']), n_frames=np.array(buf['n_frames']), fps=np.array(buf['fps']))
    part += 1
    buf = {k: [] for k in KEYS}


t0 = time.time()
for r in tqdm(todo.itertuples(), total=len(todo)):
    try:
        fr, fps = read_clip(LOCAL / r.clip_file)
        if len(fr) < N_FRAMES:
            raise ValueError(f'only {len(fr)} frames')
        whole, stages = windows_for_row(r._asdict())
        wins = [whole] + (stages or [])
        pooled, seq, cls = embed([sample_window(fr, fps, a, b)[0] for a, b in wins])
        buf['ids'].append(r.clip_id)
        buf['whole'].append(pooled[0]); buf['whole_seq'].append(seq[0]); buf['pred_class'].append(cls[0])
        buf['stages'].append(pooled[1:4] if stages else np.full((3, pooled.shape[1]), np.nan, np.float32))
        buf['has_stages'].append(stages is not None)
        buf['n_frames'].append(len(fr)); buf['fps'].append(fps)
    except Exception as e:
        errors.append((r.clip_id, repr(e)[:150]))
    if len(buf['ids']) >= 25:
        flush()
flush()
print(f'done in {(time.time() - t0) / 60:.1f} min | errors: {len(errors)}')
for e in errors[:10]:
    print('  ', e)

0 already done, 961 to go


  0%|          | 0/961 [00:00<?, ?it/s]

done in 29.0 min | errors: 0


## 7. Merge into one file (same row order as the clip list)

In [ ]:
parts = [np.load(f, allow_pickle=True) for f in sorted(FEAT_DIR.glob('part_*.npz'))]
F = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}
_, first = np.unique(F['ids'], return_index=True)
F = {k: v[np.sort(first)] for k, v in F.items()}
order = {c: i for i, c in enumerate(df.clip_id)}
idx = np.argsort([order[c] for c in F['ids']])
F = {k: v[idx] for k, v in F.items()}

out = ROOT / 'features' / 'aim_tfcovr_all.npz'
np.savez_compressed(out, **F)
meta = df.set_index('clip_id').loc[F['ids']].reset_index()
meta.to_csv(ROOT / 'features' / 'aim_tfcovr_index.csv', index=False)
print(out, {k: v.shape for k, v in F.items()})

/content/drive/MyDrive/capstone/features/aim_tfcovr_all.npz {'ids': (961,), 'whole': (961, 768), 'whole_seq': (961, 12, 768), 'stages': (961, 3, 768), 'has_stages': (961,), 'pred_class': (961,), 'n_frames': (961,), 'fps': (961,)}


## 8. Sanity checks, side by side with VideoMAE
1. **Vault identity**: test clip's nearest train clip has the same `vault_code`? VideoMAE was *below* the majority guess. A fine-grained backbone should be clearly above it.
2. **Scene bias**: train clip's nearest other train clip comes from the same source video? VideoMAE: 76%. Lower = less "which broadcast".
3. **Classifier head**: for each vault code, how consistently does AIM predict one sub-action class? High purity = the head recognises the vaults.
4. Stage separability (centred cosine).

In [ ]:
def norm(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)


def checks(path_npz, path_csv):
    G = np.load(path_npz, allow_pickle=True)
    m = pd.read_csv(path_csv, dtype={'vault_code': str})
    W = G['whole'].astype(np.float32); W = W - W.mean(0)        # centred
    tr, te = (m.split == 'train').values, (m.split == 'test').values
    code_ = m.vault_code.fillna('NA').values; vid = m.video_id.values
    sim = norm(W[te]) @ norm(W[tr]).T
    acc = (code_[tr][sim.argmax(1)] == code_[te]).mean()
    base = (code_[te] == pd.Series(code_[tr]).mode()[0]).mean()
    s = norm(W[tr]) @ norm(W[tr]).T; np.fill_diagonal(s, -9)
    same_vid = (vid[tr][s.argmax(1)] == vid[tr]).mean()
    hs = G['has_stages'].astype(bool)
    S = G['stages'][hs].astype(np.float32); S = norm(S - S.reshape(-1, S.shape[-1]).mean(0))
    within = np.mean([(S[:, i] * S[:, j]).sum(1).mean() for i, j in [(0, 1), (1, 2), (0, 2)]])
    across = np.mean([(S[:, k] @ S[:, k].T)[np.triu_indices(len(S), 1)].mean() for k in range(3)])
    return dict(nan=int(np.isnan(G['whole'].astype(np.float32)).sum()), n=len(W),
                nn_vault=acc, majority=base, same_video=same_vid, stage_within=within, stage_across=across)


res = {'VideoMAE (Day 3)': checks(ROOT / 'features' / 'videomae_k400_all.npz', ROOT / 'features' / 'videomae_k400_index.csv'),
       'AIM (TF-CoVR)': checks(out, ROOT / 'features' / 'aim_tfcovr_index.csv')}
R = pd.DataFrame(res).T
print(R.round(3).to_string())

pc = pd.Series(F['pred_class'], index=meta.index)
purity = meta.assign(pc=pc).groupby('vault_code').pc.agg(lambda s: s.value_counts().iloc[0] / len(s))
counts = meta.vault_code.value_counts()
w_purity = np.average(purity[counts.index], weights=counts.values)
print(f'\nAIM head: {pc.nunique()} distinct predicted classes | clip-weighted purity per vault code {w_purity:.3f}')
print(meta.assign(pc=pc).groupby('vault_code').pc.agg(lambda s: s.value_counts().head(3).to_dict())
      .loc[counts.index[:6]].to_string())

                  nan      n  nn_vault  majority  same_video  stage_within  stage_across
VideoMAE (Day 3)  0.0  961.0     0.164      0.26       0.760         0.196         0.198
AIM (TF-CoVR)     0.0  961.0     0.333      0.26       0.741         0.149         0.049

AIM head: 47 distinct predicted classes | clip-weighted purity per vault code 0.606
vault_code
5.80    {10: 144, 14: 25, 11: 22}
4.60       {11: 75, 15: 8, 22: 5}
5.00       {11: 64, 12: 7, 10: 7}
5.40      {10: 48, 11: 12, 27: 6}
6.30       {14: 36, 11: 4, 25: 4}
5.50       {10: 24, 12: 7, 14: 2}


In [ ]:
summary = f'''DAY 6 FEATURE EXTRACTION: TF-CoVR AIM encoder (stage-1 weights), frozen
clips: {len(F['ids'])} | with stages: {F['has_stages'].sum()} | errors: {len(errors)}

{R.round(3).to_string()}

nn_vault: test->train 1-NN same vault_code (centred features); majority = always guess the commonest code
same_video: train 1-NN from the same source video (VideoMAE Day 3: 0.76)
AIM head: {pc.nunique()} distinct classes, purity per vault code {w_purity:.3f}
'''
(ROOT / 'features' / 'day6_summary.txt').write_text(summary)
print(summary)

DAY 6 FEATURE EXTRACTION: TF-CoVR AIM encoder (stage-1 weights), frozen
clips: 961 | with stages: 886 | errors: 0

                  nan      n  nn_vault  majority  same_video  stage_within  stage_across
VideoMAE (Day 3)  0.0  961.0     0.164      0.26       0.760         0.196         0.198
AIM (TF-CoVR)     0.0  961.0     0.333      0.26       0.741         0.149         0.049

nn_vault: test->train 1-NN same vault_code (centred features); majority = always guess the commonest code
same_video: train 1-NN from the same source video (VideoMAE Day 3: 0.76)
AIM head: 47 distinct classes, purity per vault code 0.606

